In [2]:
#importing the frameworks:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [3]:

#taking a look at the data:
df = pd.read_csv("/home/bilalerremach/projects/netflix-content-trends/data/netflix_titles.csv")
print(df.head())

  show_id     type                  title         director  \
0      s1    Movie   Dick Johnson Is Dead  Kirsten Johnson   
1      s2  TV Show          Blood & Water              NaN   
2      s3  TV Show              Ganglands  Julien Leclercq   
3      s4  TV Show  Jailbirds New Orleans              NaN   
4      s5  TV Show           Kota Factory              NaN   

                                                cast        country  \
0                                                NaN  United States   
1  Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...   South Africa   
2  Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...            NaN   
3                                                NaN            NaN   
4  Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...          India   

           date_added  release_year rating   duration  \
0  September 25, 2021          2020  PG-13     90 min   
1  September 24, 2021          2021  TV-MA  2 Seasons   
2  September 24, 2021        

In [4]:
#informations about the data:
print(df.info())

#the messy columns
print(df.isna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 825.8 KB
None
show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64


In [5]:
#data cleanig...

#removing spaces from date_added colum,n
df['date_added'].str.strip()

#convert date_added to datetime
df['date_added'] = pd.to_datetime(df['date_added'], format='mixed')

#extract the year_added
df['year_added'] = df['date_added'].dt.year.astype('Int64')
print(df[['date_added', 'year_added']].head(10))

  date_added  year_added
0 2021-09-25        2021
1 2021-09-24        2021
2 2021-09-24        2021
3 2021-09-24        2021
4 2021-09-24        2021
5 2021-09-24        2021
6 2021-09-24        2021
7 2021-09-24        2021
8 2021-09-24        2021
9 2021-09-24        2021


In [6]:
#filling the empty columns with "unkown"
df['country'] = df['country'].fillna("Unknown")
df['director'] = df['director'].fillna("Unknown")
df['cast'] = df['cast'].fillna("Unknown")
print(df.isna().sum())

show_id          0
type             0
title            0
director         0
cast             0
country          0
date_added      10
release_year     0
rating           4
duration         3
listed_in        0
description      0
year_added      10
dtype: int64


In [7]:
#split the duration into number and unit
df[['duration_val', 'duration_unit']] = df['duration'].str.split(' ', n=1, expand=True)
print(df[['duration_val', 'duration_unit']].head(10))

  duration_val duration_unit
0           90           min
1            2       Seasons
2            1        Season
3            1        Season
4            2       Seasons
5            1        Season
6           91           min
7          125           min
8            9       Seasons
9          104           min


In [8]:
#checking the duplicates
print(df.duplicated().sum())

0


In [9]:
#adding a release_lag column
df['release_lag'] = df['year_added'] - df['release_year']
print(df[['title', 'year_added', 'release_year', 'release_lag']].head(10))

                              title  year_added  release_year  release_lag
0              Dick Johnson Is Dead        2021          2020            1
1                     Blood & Water        2021          2021            0
2                         Ganglands        2021          2021            0
3             Jailbirds New Orleans        2021          2021            0
4                      Kota Factory        2021          2021            0
5                     Midnight Mass        2021          2021            0
6  My Little Pony: A New Generation        2021          2021            0
7                           Sankofa        2021          1993           28
8     The Great British Baking Show        2021          2021            0
9                      The Starling        2021          2021            0


In [10]:
#converting the country and the listed_in into lists
df['country'] = df['country'].str.split(', ')
df['listed_in'] = df['listed_in'].str.split(', ')

#exploding them into one value per raw
df = df.explode('country', ignore_index=True)
df = df.explode('listed_in', ignore_index=True)
print(df[['country', 'listed_in']].head(10))

         country               listed_in
0  United States           Documentaries
1   South Africa  International TV Shows
2   South Africa               TV Dramas
3   South Africa            TV Mysteries
4        Unknown          Crime TV Shows
5        Unknown  International TV Shows
6        Unknown   TV Action & Adventure
7        Unknown              Docuseries
8        Unknown              Reality TV
9          India  International TV Shows


In [11]:
#listing all the ratings
print(df['rating'].unique())
df[df['rating'].str.contains('min')]
#we can see that the data shifted between ratings and duration

<StringArray>
[   'PG-13',    'TV-MA',       'PG',    'TV-14',    'TV-PG',     'TV-Y',
    'TV-Y7',        'R',     'TV-G',        'G',    'NC-17',   '74 min',
   '84 min',   '66 min',       'NR',        nan, 'TV-Y7-FV',       'UR']
Length: 18, dtype: str


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,year_added,duration_val,duration_unit,release_lag
14779,s5542,Movie,Louis C.K. 2017,Louis C.K.,Louis C.K.,United States,2017-04-04,2017,74 min,NaN,Movies,"Louis C.K. muses on religion, eternal love, gi...",2017,NaN,NaN,0
15409,s5795,Movie,Louis C.K.: Hilarious,Louis C.K.,Louis C.K.,United States,2016-09-16,2010,84 min,NaN,Movies,Emmy-winning comedy writer Louis C.K. brings h...,2016,NaN,NaN,6
15444,s5814,Movie,Louis C.K.: Live at the Comedy Store,Louis C.K.,Louis C.K.,United States,2016-08-15,2015,66 min,NaN,Movies,The comic puts his trademark hilarious/thought...,2016,NaN,NaN,1


In [12]:
#Filling missing values
df['rating'] = df['rating'].fillna("UR")

#migrating the misplaced durations,
df.loc[df['rating'].str.contains('min'), 'duration'] = df.loc[df['rating'].str.contains('min'), 'rating']

#Resetting the bad ratings
df.loc[df['rating'].str.contains('min'), 'rating'] = 'UR'
print(df['rating'].unique())

<StringArray>
[   'PG-13',    'TV-MA',       'PG',    'TV-14',    'TV-PG',     'TV-Y',
    'TV-Y7',        'R',     'TV-G',        'G',    'NC-17',       'UR',
       'NR', 'TV-Y7-FV']
Length: 14, dtype: str


In [13]:
#mapping the ratings
ratings_map = {
    'TV-G' : 'Kids',
    'G' : 'Kids',
    'TV-Y7-FV' : 'Kids',
    'TV-Y7' : 'Kids',
    'TV-Y' : 'Kids',
    'PG' : 'Kids',
    'TV-PG' : 'Kids',
    'PG-13' : 'Teens',
    'TV-14' : 'Teens',
    'TV-MA' : 'Adults',
    'R' : 'Adults',
    'NC-17' : 'Adults',
    'NR' : 'Adults',
    'UR' : 'Adults',
}
df['rating_category'] = df['rating'].map(ratings_map)

#info about the catigories
print(df[['rating' , 'rating_category']].head())
print("NaN : ", df['rating_category'].isna().sum())

  rating rating_category
0  PG-13           Teens
1  TV-MA          Adults
2  TV-MA          Adults
3  TV-MA          Adults
4  TV-MA          Adults
NaN :  0


In [14]:
#Movies Vs. TV Shows over time
df.groupby(['type', 'release_year']).size().unstack().fillna(0)

release_year,1925,1942,1943,1944,1945,1946,1947,1954,1955,1956,...,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021
type,,,,,,,,,,,,,,,,,,,,,
Movie,0.0,4.0,4.0,8.0,6.0,2.0,2.0,6.0,11.0,6.0,...,512.0,665.0,853.0,1215.0,1835.0,1987.0,2098.0,1603.0,1168.0,673.0
TV Show,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,...,163.0,163.0,212.0,445.0,649.0,682.0,979.0,1029.0,1142.0,767.0


In [15]:
#Content added per year
df.groupby(['type', 'year_added']).size().unstack().fillna(0)

year_added,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021
type,,,,,,,,,,,,,,
Movie,3.0,3.0,2.0,29.0,5.0,10.0,36.0,135.0,620.0,2320.0,3487.0,3887.0,3605.0,2731.0
TV Show,1.0,0.0,0.0,0.0,0.0,19.0,17.0,68.0,490.0,917.0,1056.0,1516.0,1547.0,1230.0


In [16]:
#removing the unknown data
filtered_df_countries = df.loc[df['country'] != 'Unknown']

#extracting the top 10 countries
top_countries = filtered_df_countries['country'].value_counts().head(10).index

#filtering the Dataset
filtered_df_countries = df.loc[df['country'].isin(top_countries)]

#top 10 countries and how their share changed
filtered_df_countries.groupby(['country' , 'year_added']).size().unstack().fillna(0).reindex(top_countries)

year_added,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021
country,,,,,,,,,,,,,,
United States,4.0,1.0,2.0,17.0,4.0,22.0,33.0,102.0,329.0,800.0,1091.0,1611.0,1563.0,1188.0
India,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,32.0,432.0,941.0,603.0,528.0,268.0
United Kingdom,0.0,0.0,0.0,3.0,0.0,3.0,4.0,22.0,114.0,298.0,327.0,418.0,333.0,254.0
France,0.0,0.0,0.0,5.0,0.0,0.0,0.0,16.0,61.0,131.0,154.0,192.0,228.0,129.0
Canada,0.0,0.0,0.0,0.0,0.0,1.0,5.0,13.0,59.0,155.0,164.0,154.0,208.0,118.0
Japan,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,56.0,88.0,103.0,175.0,180.0,127.0
South Korea,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,31.0,124.0,86.0,162.0,156.0,73.0
Spain,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,43.0,104.0,112.0,139.0,113.0,78.0
Germany,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,27.0,77.0,82.0,104.0,126.0,91.0


In [17]:
#extracting the top 10 genres
top_genres = df['listed_in'].value_counts().head(10).index

#filtering the Dataset
filtered_df_genres = df.loc[df['listed_in'].isin(top_genres)]

#top 10 genres overall and by year
filtered_df_genres.groupby(['listed_in', 'year_added']).size().unstack().fillna(0).reindex(top_genres)

year_added,2008,2009,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021
listed_in,,,,,,,,,,,,,
International Movies,0.0,1.0,3.0,0.0,0.0,4.0,21.0,132.0,556.0,830.0,778.0,696.0,488.0
Dramas,1.0,1.0,17.0,0.0,0.0,0.0,16.0,93.0,400.0,713.0,721.0,687.0,550.0
Comedies,0.0,0.0,0.0,3.0,1.0,4.0,15.0,57.0,210.0,345.0,491.0,503.0,352.0
International TV Shows,0.0,0.0,0.0,0.0,2.0,0.0,3.0,98.0,215.0,236.0,358.0,309.0,243.0
Action & Adventure,0.0,0.0,0.0,0.0,0.0,1.0,2.0,24.0,129.0,232.0,276.0,248.0,270.0
Documentaries,0.0,0.0,0.0,1.0,1.0,12.0,20.0,93.0,284.0,213.0,232.0,150.0,110.0
Independent Movies,1.0,0.0,0.0,0.0,0.0,0.0,7.0,44.0,158.0,235.0,257.0,203.0,135.0
TV Dramas,0.0,0.0,0.0,0.0,6.0,7.0,8.0,51.0,139.0,123.0,179.0,189.0,149.0
Children & Family Movies,0.0,0.0,3.0,1.0,2.0,6.0,17.0,25.0,94.0,117.0,185.0,241.0,154.0


In [18]:
#rating distrubition
df['rating_category'].value_counts()

rating_category
Adults    11421
Teens      7388
Kids       4945
Name: count, dtype: int64

In [25]:
#filtering the movies from df
filtered_movies = df.loc[df['type'] == 'Movie']

#cleaning the filtered_movies duration
filtered_movies['duration'] = filtered_movies['duration'].str.replace(' min', '').astype(int)
filtered_movies['duration'].describe()

count    16873.000000
mean       103.904700
std         26.231902
min          3.000000
25%         90.000000
50%        102.000000
75%        117.000000
max        312.000000
Name: duration, dtype: float64

In [28]:
#filtering the tv shows from df 
filtered_shows = df.loc[df['type'] == 'TV Show']

#cleaning the filtered_shows duration
filtered_shows['duration'] = filtered_shows['duration'].str.replace(r' Season(s)?', '',regex=True ).astype(int)
filtered_shows['duration'].value_counts()

duration
1     4628
2     1082
3      524
4      248
5      160
6       69
7       62
8       33
10      23
9       22
15       9
12       9
13       6
11       4
17       2
Name: count, dtype: int64